# Goal-Conditioned Analog AI Sizing - Kaggle Version (Drive Download)
This notebook is optimized for **Kaggle** to allow Silent Background Training. It downloads the heavy 5.5GB LUT files directly from your Google Drive so you don't have to upload them to Kaggle!

## Setup Instructions
1. **On Google Drive:** Make the two LUT files (`TSMC_fast_65nm_nch.pkl` and `TSMC_fast_65nm_pch.pkl`) **Public** (Anyone with the link can view).
2. **Get the IDs:** Copy the file ID from their sharing links and paste them into the code cell below.
3. **Upload Scripts to Kaggle:** Zip your `Analog_AI_Optimization` folder (WITHOUT the heavy `tech_luts` folder) and upload it as a Kaggle Dataset (it will take 1 second since it's just code).
4. Click **Save Version -> Save & Run All (Commit)**.

In [ ]:
!pip install stable-baselines3[extra] gymnasium numpy scipy tensorboard gdown

## 1. Download LUTs from Google Drive (1-2 Gbps Speed)

In [ ]:
import os

# Create tech_luts directory in the working folder
os.makedirs('/kaggle/working/tech_luts', exist_ok=True)
os.chdir('/kaggle/working/tech_luts')

# --- PASTE YOUR GOOGLE DRIVE FILE IDs HERE ---
# Example link: https://drive.google.com/file/d/1XyZ...abc/view
# The ID is the random text between /d/ and /view
NCH_FILE_ID = '10YDQaHxinCaGA3ayCe0LMTS_mCYx4mkA'
PCH_FILE_ID = '1oEEmU5b3nJONe9dolYPGp8w9DacyU8Qf'

print("Downloading NCH LUT...")
!gdown --id {NCH_FILE_ID} -O TSMC_fast_65nm_nch.pkl

print("Downloading PCH LUT...")
!gdown --id {PCH_FILE_ID} -O TSMC_fast_65nm_pch.pkl

os.chdir('/kaggle/working/')
print("Download Complete!")

In [ ]:
import sys

# --- KAGGLE SETUP ---
# Change this to the exact name you gave your code dataset
DATASET_NAME = 'analog-ai-code'

PROJECT_PATH = f'/kaggle/input/{DATASET_NAME}/'
sys.path.append(PROJECT_PATH)

WORKING_DIR = '/kaggle/working/'

## 2. Load the Core Engine and LUTs

In [ ]:
from tech_luts.lut_utils import LUT
from core.device_model import DeviceModel
from circuits.ota5t import OTA5T
from optimizer.rl_environment import OTA5tGymEnv

print("Loading LUTs into memory...")
nch_path = os.path.join(WORKING_DIR, 'tech_luts', 'TSMC_fast_65nm_nch.pkl')
pch_path = os.path.join(WORKING_DIR, 'tech_luts', 'TSMC_fast_65nm_pch.pkl')

nch = LUT(nch_path)
pch = LUT(pch_path)

dm = DeviceModel(nch, pch)
ota = OTA5T(dm, vdd=1.2, cl=1e-12)
ota.Vicm = 0.5
print("Physics Engine Ready!")

In [ ]:
bounds = [
    (60e-9, 1.0e-6),  # L1
    (5.0, 25.0),      # gmid1
    (60e-9, 1.0e-6),  # L3
    (5.0, 25.0),      # gmid3
    (60e-9, 1.0e-6),  # L5
    (5.0, 25.0),      # gmid5
    (10e-6, 500e-6)   # Itail
]

env = OTA5tGymEnv(ota, bounds=bounds, max_steps=200)

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CheckpointCallback

checkpoint_callback = CheckpointCallback(
    save_freq=50000, 
    save_path=os.path.join(WORKING_DIR, 'checkpoints'),
    name_prefix='kaggle_ppo_model'
)

In [ ]:
tb_log_dir = os.path.join(WORKING_DIR, 'ppo_ota_tensorboard')

model = PPO("MlpPolicy", env, verbose=1, learning_rate=0.0005, batch_size=256, tensorboard_log=tb_log_dir)

print("Starting 500k Steps Training on Kaggle...")
model.learn(total_timesteps=500000, callback=checkpoint_callback)

final_model_path = os.path.join(WORKING_DIR, 'universal_ppo_agent_65nm')
model.save(final_model_path)
print(f"Training Complete! Model saved to {final_model_path}.zip")